In [ ]:
import math

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import matplotlib.pyplot as plt

print(torch.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

In [ ]:
# !! 旧 token 已经写进过文件, 务必去 HuggingFace settings 撤销并重新生成
# 推荐: 终端执行 `huggingface-cli login`, 或从环境变量读取, 不要硬编码进 notebook
# import os
# assert os.getenv("HF_TOKEN"), "run: export HF_TOKEN=... before starting jupyter"

# Create Dataset

In [3]:
from datasets import load_from_disk
import pandas as pd

ds = load_from_disk("./snli-ve")

Loading dataset from disk:   0%|          | 0/147 [00:00<?, ?it/s]

# Using Dataloader loading data from dataset- image-text

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess(examples):
    inputs = tokenizer(
        examples["hypothesis"],
        truncation=True,
        max_length=128
    )
    inputs["pixel_values"] = [
        image_transform(img.convert("RGB"))
        for img in examples["image"]
    ]
    inputs["labels"] = examples["label"]
    return inputs

train_ds = ds["train"].select(range(1000))

train_ds = train_ds.filter(lambda label: label != -1, input_columns=["label"])
train_ds.set_transform(preprocess)

text_collator = DataCollatorWithPadding(tokenizer)

def multimodal_collate_fn(features):
    images = [f.pop("pixel_values") for f in features]
    batch = text_collator(features)          # pad input_ids / attention_mask
    batch["pixel_values"] = torch.stack(images)
    return batch

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=multimodal_collate_fn,
    num_workers=4, 
    pin_memory=True
)

In [22]:
batch = next(iter(train_loader))

print(batch.keys())
for k, v in batch.items():
    print(k, type(v), getattr(v, "shape", None), getattr(v, "dtype", None))

KeysView({'labels': tensor([2, 0, 0, 2, 1, 0, 2, 0, 1, 1, 0, 1, 0, 1, 0, 2, 0, 0, 0, 0, 2, 0, 1, 0,
        2, 1, 0, 0, 0, 0, 2, 1]), 'input_ids': tensor([[  101,  1037,  2158, 14020,  2046,  1996,  3712,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0],
        [  101,  1037,  2158,  4147,  1037,  3756,  3808, 17447,  5610,  2015,
          1998,  2515,  5957,  2147,   102,     0,     0],
        [  101,  1996,  2611,  2001,  2893,  4954,  1012,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0],
        [  101,  1037,  9074,  3084,  1037,  3819, 12313, 10055,  1012,   102,
             0,     0,     0,     0,     0,     0,     0],
        [  101,  2048,  2273,  3868,  2006, 18580,  2024,  2200,  2485,  2000,
          2028,  2178,  1012,   102,     0,     0,     0],
        [  101,  1037,  2177,  3564,  2006,  1037,  2600,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0],
        [  101,  1996,  3

# Custom CAMC-NET Model

In [ ]:
import torch.nn as nn
from transformers import ViTModel, BertModel

class ImageEncoder(nn.Module):
    def __init__(self, model="google/vit-base-patch16-224"):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
       
        img_emb = outputs.last_hidden_state   # [B, 1+N_patch, 768]
        return img_emb


class TextEncoder(nn.Module):
    def __init__(self, model="bert-base-uncased"):
        super().__init__()
        self.bert = BertModel.from_pretrained(model)

    def forward(self, input_ids, attention_mask=None):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_emb = output.last_hidden_state    # [B, N_txt, 768]
        return txt_emb

In [ ]:
class ContradictionAwareLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm_t1 = nn.LayerNorm(hidden_dim)
        self.ffn_t = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.norm_t2 = nn.LayerNorm(hidden_dim)

    def forward(self, img_emb, txt_emb, img_mask=None):
       
        cross_out, attn_weights = self.cross_attn(
            query=txt_emb,
            key=img_emb,
            value=img_emb,
            key_padding_mask=img_mask
        )  
        txt_emb = self.norm_t1(txt_emb + cross_out)

        ffn_out = self.ffn_t(txt_emb)
        txt_emb = self.norm_t2(txt_emb + ffn_out)

        return txt_emb, attn_weights

In [ ]:
class ContradictionAwareEncoder(nn.Module):
    def __init__(self, hidden_dim, num_heads=8, num_layers=6):
        super().__init__()
        self.img_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layers = nn.ModuleList([
            ContradictionAwareLayer(hidden_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, img_emb, txt_emb):
        img_emb = self.img_proj(img_emb)   # ablation: img_proj

        all_attn_weights = []
        for layer in self.layers:
            txt_emb, attn_weight = layer(img_emb, txt_emb)
            all_attn_weights.append(attn_weight)   

        text_pool = txt_emb[:, 0]   
        img_pool = img_emb[:, 0]    

        fusion_feature = torch.cat([
            img_pool,
            text_pool,
            torch.abs(text_pool - img_pool),
            text_pool * img_pool
        ], dim=-1)                  # [B, 4*hidden]

        return fusion_feature, all_attn_weights

In [ ]:
class CAMC(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8,
                 num_layers=6, num_classes=3):
        super().__init__()
        self.ie = ImageEncoder()
        self.te = TextEncoder()
        self.encoder = ContradictionAwareEncoder(
            hidden_dim, num_heads, num_layers
        )


        for p in self.ie.parameters():
            p.requires_grad = False
        for p in self.te.parameters():
            p.requires_grad = False
        self.ie.eval()
        self.te.eval()

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask=None):
        with torch.no_grad():  
            img_emb = self.ie(pixel_values)
            txt_emb = self.te(input_ids, attention_mask)

        fusion_feature, all_attn_weights = self.encoder(img_emb, txt_emb)
        logits = self.classifier(fusion_feature)

        return logits, all_attn_weights

    def train(self, mode=True):
        super().train(mode)   
        self.ie.eval()        
        self.te.eval()
        return self

# Sanity Check

In [ ]:

model = CAMC(hidden_dim=768).to(device)   

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable/1e6:.1f}M / total: {total/1e6:.1f}M")

model.train()
print("encoders stay eval:", not model.ie.training, not model.te.training)  # True True 才对

batch = next(iter(train_loader))
with torch.no_grad():
    logits, attn = model(
        batch["pixel_values"].to(device),
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )
print("logits:", logits.shape)                        # [32, 3]
print("attn layers:", len(attn), attn[0].shape)       # 6 layers, [32, N_txt, N_img]